## Setup

In [0]:
%pip install -r requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
from mlflow.models import ModelConfig
from pypdf import PdfReader
import pandas as pd
import re

In [0]:
dataset_conf_path = "conf/dataset.yml"
dataset_conf = ModelConfig(development_config=dataset_conf_path).get("dataset")
last_content_page = 27

In [0]:
main_pdf_path = dataset_conf.get("documents").get("main_pdf").get("full_path")
main_pdf_reader = PdfReader(main_pdf_path)

## Read PDF and Perform Chunking

In [0]:
import re
from typing import Optional, List, Dict

def get_page_number(text: str) -> str:
    """
    Extracts the page number from the given text.

    Args:
        text (str): The text from which to extract the page number.

    Returns:
        str: The extracted page number, or "-99" if no page number is found.
    """
    match = re.match(r' (\d+)', text)
    if match:
        page_number = match.group(1)
        return page_number.strip()
    else:
        return "-99"

def get_chapter(text: str) -> Optional[str]:
    """
    Extracts the chapter title from the given text.

    Args:
        text (str): The text from which to extract the chapter title.

    Returns:
        Optional[str]: The extracted chapter title, or None if no chapter title is found.
    """
    match = re.search(r'\n\s*(\d+\.\s+.+)', text)
    if match:
        chapter = match.group(1).strip()
        return chapter
    else:
        return None

def get_footnotes(text: str) -> str:
    """
    Extracts the footnotes from the given text.

    Args:
        text (str): The text from which to extract the footnotes.

    Returns:
        str: The extracted footnotes, or an empty string if no footnotes are found.
    """
    pattern = r' {10}\n(.*)'
    match = re.search(pattern, text, re.DOTALL)

    if match:
        citation = match.group(1).strip()
        return citation
    else:
        return ""

def clean_text(text: str, footnotes: str, page_number: str) -> str:
    """
    Cleans the text by removing footnotes and page numbers.

    Args:
        text (str): The text to be cleaned.
        footnotes (str): The footnotes to be removed from the text.
        page_number (str): The page number to be removed from the text.

    Returns:
        str: The cleaned text.
    """
    text = text.replace(footnotes, "")
    text = text.replace(f"{page_number} \n \n", "")
    return text

def split_footnotes(chapter_footnotes: List[str]) -> List[str]:
    """
    Splits the footnotes into individual footnotes.

    Args:
        chapter_footnotes (List[str]): The list of footnotes to be split.

    Returns:
        List[str]: The list of individual footnotes.
    """
    split_footnotes = []
    for footnote in chapter_footnotes:
        parts = re.split(r'\n(?=\d)', footnote)
        stripped_parts = [part.strip() for part in parts]
        split_footnotes.extend(stripped_parts)
    return split_footnotes

In [0]:
parsed_content = [page_content.extract_text() for page_content in main_pdf_reader.pages]

In [0]:
current_chapter = "0. Preface"
previous_chapter = current_chapter

min_page = 1
chapter_text = ""
chapter_info = {}
chapter_footnotes = []

for text in parsed_content[:last_content_page]:
    page_number = int(get_page_number(text))
    chapter = get_chapter(text)
    footnotes = get_footnotes(text)

    text = clean_text(text, footnotes, page_number)

    ## If page has a new chapter
    if chapter:
        previous_chapter = current_chapter
        current_chapter = chapter

        if current_chapter in text:
            leftover_text, new_chapter_text = text.split(current_chapter, 1)
        else:
            leftover_text, new_chapter_text = text, ""

        chapter_text += leftover_text
        max_page = page_number if leftover_text.strip() else page_number - 1

        ## Save previous chapter information
        chapter_info[previous_chapter] = {
            "chapter_text": chapter_text.strip(),
            "min_page": min_page,
            "max_page": max_page,
            "footnotes": split_footnotes(chapter_footnotes)
        }

        ## Start of a New Chapter
        chapter_text = new_chapter_text
        chapter_footnotes = []
        min_page = page_number
        
    else:
        chapter_text += text

    if footnotes:
      chapter_footnotes.append(footnotes.strip())

# Handle last chapter after loop
max_page = page_number if chapter_text.strip() else page_number - 1
chapter_info[current_chapter] = {
    "chapter_text": chapter_text.strip(),
    "min_page": min_page,
    "max_page": max_page,
    "footnotes": split_footnotes(chapter_footnotes)
}

## Save as Table

In [0]:
chunk_table_name = dataset_conf.get("tables").get("chapter_chunk_table").get("full_path")

In [0]:
# Convert chapter_info to a pandas dataframe
chapter_info_df = pd.DataFrame.from_dict(chapter_info, orient='index')

# # Save as a delta table, overwriting the entire table including the schema
chapter_info_spark_df = spark.createDataFrame(chapter_info_df.reset_index().rename(columns={'index': 'chapter'}))
chapter_info_spark_df.write.format("delta").mode("overwrite").saveAsTable(chunk_table_name)

# Enable Change Data Feed for Vector Index creation later
spark.sql(f"ALTER TABLE {chunk_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")